In [0]:
from pyspark.sql import functions as F

df = spark.table("fmcg.gold.vw_fact_orders_denorm")
df.printSchema()
df.count()

root
 |-- date: date (nullable = true)
 |-- product_code: string (nullable = true)
 |-- customer_code: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- month_short_name: string (nullable = true)
 |-- quarter: string (nullable = true)
 |-- year_quarter: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- variant: string (nullable = true)
 |-- sold_quantity: long (nullable = true)
 |-- price_inr: long (nullable = true)
 |-- total_amount_inr: long (nullable = true)



101015

Data quality checks

In [0]:
dq_check = df.select(
    F.count(F.when(F.col("sold_quantity").isNull(), 1)).alias("null_qty"), #null values
    F.count(F.when(F.col("sold_quantity") < 0, 1)).alias("negative_qty"),#negative values
    F.count(F.when(F.col("price_inr").isNull(), 1)).alias("null_price"),# null prices
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
)
display(dq_check)

null_qty,negative_qty,null_price,min_date,max_date
0,0,0,2024-01-01,2025-12-01


In [0]:
# Agrregate the data by product, channel, date

agg_df = (
    df.groupBy("product_code", "channel", "date")
      .agg(
          F.sum("sold_quantity").alias("sold_quantity"),
          F.first("division").alias("division"),
          F.first("category").alias("category"),
          F.first("product").alias("product"),
          F.avg("price_inr").alias("price_inr"),   # price is static per year, avg is safe
          F.first("market").alias("market"),
          F.first("platform").alias("platform")
      )
      .orderBy("product_code", "channel", "date")
)

display(agg_df)

product_code,channel,date,sold_quantity,division,category,product,price_inr,market,platform
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-07-01,76219,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-08-01,75681,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-09-01,80900,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-10-01,75013,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-11-01,71409,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Acquisition,2025-12-01,9061,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),281.0,India,Sports Bar
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-07-01,124782,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-08-01,120725,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-09-01,121121,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,Acquisition,2025-10-01,114977,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),100.0,India,Sports Bar


In [0]:
(agg_df.write
    .mode("overwrite") #This table is meant to be rebuilt from the source each time the orchestration job runs
    .option("overwriteSchema", "true")
    .saveAsTable("fmcg.gold.fact_sales_monthly_agg")
)#writing to a delta table in the gold layer